> 在线讲义：[第 25 章 · LQR 与 Riccati：从动态规划推出 Riccati 方程，有限 / 无限时域与可控性](https://wikieden.github.io/math-and-eval-stats/lessons/25-lqr-riccati.html) · 本书《数学与实验统计》

# 第 25 章 · LQR 与 Riccati：从动态规划推出 Riccati 方程，有限 / 无限时域与可控性

配套讲义：仓库里 `math-and-eval-stats/notes/lessons/25-lqr-riccati.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。只用 NumPy / SciPy（Colab 自带），不需要 GPU。练习题不在这里算，先在讲义里自己做。

In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)  # 打印保留 4 位小数

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一行代码、跑出来是什么。

| 讲义 | 公式 | 写法 | 跑出来 |
|---|---|---|---|
| 第一步 往回推 | Riccati 差分 | `finite_horizon(10)` | P₉ = 1.72，P₀ = 1.9522 |
| 第二步 无限时域 | DARE 正根 | `dare_scalar()` | 1.9522，K = 0.7935 |
| 第三步 总代价 | x₀ᵀP₀x₀ | `rollout_cost(2.0, ks)` | 7.8089 |
| 第四步 调 R | DARE(r) | `dare_scalar(r=0.01)` | K = 1.1883 |
| 第五步 可控性 | 秩 | `ctrb_rank(A, B)` | 2 / 1 |

例子：标量系统 x_{k+1} = a·x_k + b·u_k（a = 1.2 不稳定，b = 1），代价 Σ q·x² + r·u²（q = r = 1），终端代价 q_f·x_N²。
用标量是为了每一步都能手算；矩阵版就是把乘法换成矩阵乘法。
里程碑 ME-M20 的"矩阵系统的有限时域 Riccati 和 scipy.linalg.solve_discrete_are 对数"留给自己做。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
import numpy as np
A = 1.2
B = 1.0
Q = 1.0
R = 1.0

**函数 `riccati_step`**：动态规划往回推一步：已知下一步的值函数 p·x²，求这一步的 (p, K)。

In [ ]:
def riccati_step(p: float, a: float = A, b: float = B, q: float = Q, r: float = R) -> tuple[float, float]:
    """动态规划往回推一步：已知下一步的值函数 p·x²，求这一步的 (p, K)。
    min_u [q x² + r u² + p (a x + b u)²] → u = −K x，K = a b p / (r + b² p)，新 p = q + a² p − (a b p)² / (r + b² p)。"""
    k = a * b * p / (r + b * b * p)
    return q + a * a * p - (a * b * p) ** 2 / (r + b * b * p), k

**函数 `finite_horizon`**：从终端 p_N = q_f 往回推 n 步，返回按时间顺序的 [p_0, …, p_N] 和 [K_0, …, K_{N−1}]。

In [ ]:
def finite_horizon(n: int, qf: float = 1.0, **kw) -> tuple[list[float], list[float]]:
    """从终端 p_N = q_f 往回推 n 步，返回按时间顺序的 [p_0, …, p_N] 和 [K_0, …, K_{N−1}]。"""
    ps, ks, p = [qf], [], qf
    for _ in range(n):
        p, k = riccati_step(p, **kw)
        ps.append(p)
        ks.append(k)
    return ps[::-1], ks[::-1]

**函数 `dare_scalar`**：无限时域：p = q + a²p − a²b²p²/(r + b²p) 化成 b²p² + (r − a²r − q b²)p − q r = 0，取正根。

In [ ]:
def dare_scalar(a: float = A, b: float = B, q: float = Q, r: float = R) -> float:
    """无限时域：p = q + a²p − a²b²p²/(r + b²p) 化成 b²p² + (r − a²r − q b²)p − q r = 0，取正根。"""
    c2, c1, c0 = b * b, r - a * a * r - q * b * b, -q * r
    return (-c1 + np.sqrt(c1 * c1 - 4 * c2 * c0)) / (2 * c2)

**函数 `gain`**：小工具，把结果整理成好读的样子再打印。

In [ ]:
def gain(p: float, a: float = A, b: float = B, r: float = R) -> float:
    return a * b * p / (r + b * b * p)

**函数 `rollout_cost`**：按增益序列闭环跑一遍，把每一步代价加起来（含终端）。

In [ ]:
def rollout_cost(x0: float, ks: list[float], qf: float = 1.0) -> float:
    """按增益序列闭环跑一遍，把每一步代价加起来（含终端）。"""
    x, c = x0, 0.0
    for k in ks:
        u = -k * x
        c += Q * x * x + R * u * u
        x = A * x + B * u
    return c + qf * x * x

**函数 `ctrb_rank`**：可控性矩阵 [B, AB, …, A^{n−1}B] 的秩；等于 n 才能把状态推到任意地方。

In [ ]:
def ctrb_rank(a: np.ndarray, b: np.ndarray) -> int:
    """可控性矩阵 [B, AB, …, A^{n−1}B] 的秩；等于 n 才能把状态推到任意地方。"""
    n = a.shape[0]
    cols = [b]
    for _ in range(n - 1):
        cols.append(a @ cols[-1])
    return int(np.linalg.matrix_rank(np.hstack(cols)))

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步 动态规划往回推：值函数一直是 p_k·x²")
ps, ks = finite_horizon(10)
print(f"  N = 10，p_N = 1：p_9 = {ps[9]:.4f}，p_8 = {ps[8]:.4f}，p_5 = {ps[5]:.4f}，p_0 = {ps[0]:.4f}")
print(f"  增益 K_9 = {ks[9]:.4f}，K_0 = {ks[0]:.4f}")

In [ ]:
print("\n第二步 无限时域：p 收敛到代数 Riccati 方程的正根")
p = dare_scalar()
print(f"  p∞ = {p:.4f}，K∞ = {gain(p):.4f}，闭环 a − bK = {A - B * gain(p):.4f}（|·| < 1，稳了）")
for n in (1, 2, 5, 10):
    print(f"  往回推 {n} 步：p_0 = {finite_horizon(n)[0][0]:.4f}")

In [ ]:
print("\n第三步 p_0·x0² 就是最优总代价")
x0 = 2.0
print(f"  x0 = {x0}：p_0·x0² = {ps[0] * x0 * x0:.4f}，按 K_k 闭环跑一遍加起来 = {rollout_cost(x0, ks):.4f}")
for k in (0.6, 1.0, 1.2):
    print(f"  换成固定增益 K = {k}：{rollout_cost(x0, [k] * 10):.4f}（都更大）")

In [ ]:
print("\n第四步 r 的作用")
for r in (0.01, 1.0, 100.0):
    pr = dare_scalar(r=r)
    kr = gain(pr, r=r)
    print(f"  r = {r}：p∞ = {pr:.4f}，K = {kr:.4f}，闭环极点 {A - B * kr:.4f}")

In [ ]:
print("\n第五步 可控性：控制量够不着的不稳定模态，LQR 也救不了")
a = np.array([[1.0, 0.1], [0.0, 1.0]])
print(f"  双积分器，B = [0, 0.1]：秩 {ctrb_rank(a, np.array([[0.0], [0.1]]))}（可控）")
a2 = np.diag([1.2, 0.5])
print(f"  A = diag(1.2, 0.5)，B = [0, 1]：秩 {ctrb_rank(a2, np.array([[0.0], [1.0]]))}（1.2 那个模态推不动，会一直发散）")

## 3. 代码里才看得到的

1. **列表倒序**：往回推时先算出 P_N、P_{N−1}…，代码最后 `[::-1]` 翻成按时间顺序，K_k 和 P_{k+1} 才对得上。
2. **增益用的是下一步的 P**：`riccati_step(p)` 里同一个 p 同时算 K 和新 p，这个 p 是 P_{k+1}。
3. **标量 DARE 取正根**：二次方程两个根一正一负，负根不对应任何"代价"，代码直接取 `+sqrt`。

## 4. 自己改一改

1. 把 Q_f 改成 100，看 P_k 要往回推几步才收敛到同一个 P∞。
2. 把 a 改成 0.8（自己就稳定），R 调到 1000，K 会变成多少？为什么这时可以几乎不控制？

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 倒着排行程，值函数一直是二次型，Riccati 往回推；
> 推到不动就是 DARE，常数增益 K 拿去用；
> xᵀPx 是报价单，别的增益只会更贵；
> Q 比 R 定激进；够不着的不稳模态谁也救不了。